# FMSE Lab 06 - Architecture Trade Study Workbench

**Course:** FMSE 2026 · **Module 6:** Architecture Patterns and Trade Studies · **Phase:** Architect · **Tier:** Engineer

Course home and progress: [agentic-ai.es/academy/fmse/learn/m06-architecture-trade-studies](https://agentic-ai.es/academy/fmse/learn/m06-architecture-trade-studies).

## 0. Mission and competency target

**Mission.** Design three architectures for a high-volume support workflow with strict security and cost constraints. Select one only after measured/estimated trade analysis.

**Guided lab.** Students complete candidate matrices and sensitivity plots from supplied benchmark data.

**Competency target (Module 6 outcomes):**
- Compare single-call, deterministic pipeline, prompt chain, workflow/orchestrator, tool-using agent, and multi-agent patterns.
- Create Pugh/weighted trade studies with sensitivity analysis.
- Identify trust boundaries, deterministic boundaries, and failure containment zones.
- Produce Architecture Decision Records tied to requirements.

**Scaffolding:** about 25% guided, 75% independent. This is an **design challenge**: the guided part gives you a working reference path; the challenge does not.

## 1. Requirements and acceptance criteria

Public validators check these requirements. They report which requirement failed and why — never the expected implementation.

| ID | Requirement |
| --- | --- |
| ARC-01 | >=3 credible alternatives |
| ARC-02 | Criteria trace to requirements |
| ARC-03 | Weights sum to 1 |
| ARC-04 | Sensitivity analysis executed |
| ARC-05 | Decision includes reversal/revisit conditions |

**Competency gate (portal):** Architecture defense PASS. Quiz >= 80%, the guided lab, the artifact and your reflection are also required. A critical requirement cannot be offset by other scores.

## 2. Environment setup

In [ ]:
# Idempotent: safe to re-run, and it never prints secrets.
LAB_ID = "lab-06"
LABKIT_VERSION = "1.0.0"
LABKIT_SOURCES = [
    "https://agentic-ai.es/academy/fmse/labs",  # agentic-ai.es mirror
    "https://raw.githubusercontent.com/farithjhg/fmse-course-labs/main",  # GitHub (public repositories only)
]
import hashlib, json, os, pathlib, sys, urllib.request


def _fmse_bootstrap():
    """Make fmse_labkit importable: installed copy, local checkout, cached download, or verified download."""
    try:
        import fmse_labkit
        if fmse_labkit.__version__ == LABKIT_VERSION:
            return "fmse_labkit already loaded"
    except ImportError:
        pass
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "fmse_labkit" / "__init__.py").exists():
            sys.path.insert(0, str(folder))
            return f"using the labkit in {folder}"
    target = pathlib.Path("fmse_labkit_dist").resolve()
    marker = target / f".labkit-{LABKIT_VERSION}"
    if marker.exists():
        sys.path.insert(0, str(target))
        return "using the previously downloaded labkit"
    problems = []
    for base in [os.environ.get("FMSE_LABKIT_URL")] + LABKIT_SOURCES:
        if not base:
            continue
        base = base.rstrip("/")
        try:
            manifest = json.loads(urllib.request.urlopen(base + "/manifest.json", timeout=20).read())
            if manifest.get("labkit_version") != LABKIT_VERSION:
                raise RuntimeError(f"mirror serves labkit {manifest.get('labkit_version')}")
            for entry in manifest["files"]:
                if not entry["path"].startswith("fmse_labkit/"):
                    continue
                data = urllib.request.urlopen(f"{base}/{entry['path']}", timeout=20).read()
                if hashlib.sha256(data).hexdigest() != entry["sha256"]:
                    raise RuntimeError(f"integrity check failed for {entry['path']}")
                dest = target / entry["path"]
                dest.parent.mkdir(parents=True, exist_ok=True)
                dest.write_bytes(data)
            marker.write_text("ok")
            sys.path.insert(0, str(target))
            return f"downloaded and verified the labkit from {base}"
        except Exception as exc:  # try the next source
            problems.append(f"{base} ({type(exc).__name__})")
    raise RuntimeError(
        "Could not load fmse_labkit from: " + "; ".join(problems) + ". "
        "Recovery: download the labs folder from agentic-ai.es/academy/fmse/resources, upload the "
        "fmse_labkit folder next to this notebook (Files panel), and re-run this cell."
    )


print(_fmse_bootstrap())
import fmse_labkit as fmse

print(f"fmse_labkit {fmse.__version__} ready for {LAB_ID} ({fmse.lab_spec(LAB_ID).title})")

## 3. Secrets check

This lab runs fully offline; no provider key is needed. Keys you use elsewhere belong in Colab Secrets and are never printed.

In [ ]:
fmse.secrets_check()

## 4. Guided experiment

Supplied, estimated benchmark data for six architecture patterns in a high-volume support workflow. `weighted_totals`, `winner` and `perturb` implement the course's trade-study conventions.

In [ ]:
from fmse_labkit.labs import lab06
import json

guided = fmse.GuidedLog(LAB_ID, required_steps=["matrix_completed", "sensitivity_run"])
print(json.dumps(lab06.SCENARIO, indent=2))
for p, row in lab06.BENCHMARK.items():
    print(f"{p:<24} {row}")

## 5. Predict before you run

Which pattern will win a matrix weighted towards security and cost — and does the most accurate pattern win? Write your prediction, then run the experiment.

In [ ]:
my_prediction = ""
guided.predict("prediction", my_prediction)

In [ ]:
criteria = [{"id": "sec", "weight": 0.35}, {"id": "cost", "weight": 0.35}, {"id": "acc", "weight": 0.30}]
scores = {"pipeline": {"sec": 5, "cost": 5, "acc": 3}, "orchestrator": {"sec": 4, "cost": 4, "acc": 4}, "agent": {"sec": 2, "cost": 1, "acc": 5}}
print("totals:", lab06.weighted_totals(scores, criteria), "winner:", lab06.winner(scores, criteria))
guided.step("matrix_completed")
for cid in ("sec", "cost"):
    for d in (0.2, -0.2):
        print(cid, f"{d:+.0%}", "->", lab06.winner(scores, lab06.perturb(criteria, cid, d)))
guided.step("sensitivity_run")
guided.outcome("prediction", "see output above")

**Worksheet.** At least one full sentence per field; these observations are guided-lab evidence.

In [ ]:
worksheet = {
    "matrix_observation": "",
    "sensitivity_observation": "",
    "prediction_check": "",
}
guided.observe(worksheet)

## 6. Challenge

**Three architectures for a high-volume support workflow.** Design three architectures for a high-volume support workflow with strict security and cost constraints. Select one only after measured/estimated trade analysis.

Constraints:
- Criteria must trace to the scenario requirement ids.
- Sensitivity: perturb the two highest weights by +/-20% and renormalize.

In [ ]:
my_study = {
    "alternatives": [],  # {"id", "name", "pattern" (see lab06.PATTERNS), "description"}
    "criteria": [],  # {"id", "name", "weight", "requirements": [scenario requirement ids]}
    "scores": {},  # alternative id -> {criterion id: 1-5}
    "sensitivity": [],  # {"criterion", "delta": +0.2/-0.2, "winner"} for the two highest weights
    "decision": {"selected": "", "context": "", "rationale": "", "consequences": "", "revisit_triggers": []},
}

## 7. Public validation

In [ ]:
submission = my_study
result = fmse.check_public(LAB_ID, submission)
result.show()

## 8. Robustness / adversarial probes

Extra adversarial conditions beyond the graded suite. They do not change your score; they show where your solution is brittle.

In [ ]:
probe_results = fmse.probe(LAB_ID, submission)

## 9. Engineering reflection

Graded in the portal and stored with your artifact. Draft it here if useful:
- **Architecture defense:** Why does the selected architecture exist, and why were the others rejected? Use your trade study as evidence.
- **Trade-offs:** What does your choice give up, and under which requirement changes would you reverse it?
- **Remaining risks:** Which trust boundary or failure containment zone worries you most?

In [ ]:
reflection_draft = {"architecture_defense": "", "tradeoffs": "", "remaining_risks": ""}

## 10. Export artifact / completion result

Exports the **Architecture Specification + Pugh/Trade Study + ADRs** artifact and prints a completion record. Paste the record into the Module 6 page on agentic-ai.es (**Import lab result**). The record is a learning-workflow document, not a signed certificate.

In [ ]:
artifact = fmse.export_artifact(LAB_ID, my_study, title="Architecture Specification + Pugh/Trade Study + ADRs", fmt="json")
record = fmse.make_completion_record(
    LAB_ID,
    result,
    {"reflection_draft": reflection_draft, **{}},
    artifact=artifact,
    guided=guided,
    probes=probe_results,
)
fmse.show_record(record)